# Notebook B v4: gold_research_person_signals

**Run order:** After Notebook A v4.

## v4 changes vs v3

Added to the view SELECT output (needed by gold_person_signal_scores):
- `birth_year`, `death_year`, `sex` — from gold_person_life
- `expected_end_year` — computed in the timeline CTE, now exposed
- `earliest_marriage_year` — from gold_person_fact_summary (for IMPRECISE_DATES applicability)
- `num_child_births`, `num_marriages`, `event_count` — already present, confirmed exposed
- `total_expected_censuses` — sum of expected census year flags (for MISSING_CENSUS_COVERAGE applicability)
- `has_any_transcript` — boolean flag (for FACT_CONFLICT / TRANSCRIPT_ONLY applicability)
- `has_citations` — boolean flag (for UNCOVERED_SOURCES applicability)

No changes to signal detection logic.

`gold_research_person_signals_pivoted` is **retired** — replaced by
`gold_person_signal_scores` (Notebook D v4) which generates the long-format
per-person × signal table directly.


In [0]:

%sql
-- ============================================================
-- NOTEBOOK B v4: gold_research_person_signals
--
-- v4 changes vs v3:
--   ADDED to SELECT output:
--     birth_year, death_year, sex          — from gold_person_life
--     expected_end_year                    — from timeline CTE (was internal only)
--     earliest_marriage_year               — from gold_person_fact_summary
--     total_expected_censuses              — sum of expected_N flags
--     has_any_transcript                   — for FACT_CONFLICT applicability
--     has_citations                        — for UNCOVERED_SOURCES applicability
--   RETIRED: gold_research_person_signals_pivoted
--            (replaced by gold_person_signal_scores in Notebook D v4)
--
-- All signal detection logic is unchanged from v3.
-- ============================================================

CREATE OR REPLACE VIEW genealogy.gold_research_person_signals AS

WITH timeline AS (
  SELECT
    t.person_gedcom_id,
    MAX(p.sex)                                    AS sex,
    MAX(p.birth_year)                             AS birth_year,
    MAX(p.death_year)                             AS death_year,
    MIN(event_year_parsed)                        AS first_event_year,
    MAX(event_year_parsed)                        AS last_event_year,
    MAX(age_years)                                AS max_event_age,
    COALESCE(
      MAX(p.death_year),
      LEAST(MAX(p.birth_year) + 80, year(current_date)),
      LEAST(MIN(event_year_parsed) + 80, year(current_date))
    )                                             AS expected_end_year,
    (
      COALESCE(
        MAX(p.death_year),
        LEAST(MAX(p.birth_year) + 80, year(current_date)),
        LEAST(MIN(event_year_parsed) + 80, year(current_date))
      ) - MIN(event_year_parsed) + 1
    )                                             AS effective_span_years,
    COUNT(*)                                      AS event_count
  FROM genealogy.gold_person_event_timeline t
  JOIN genealogy.gold_person_life p ON p.person_gedcom_id = t.person_gedcom_id
  WHERE event_year_parsed IS NOT NULL
  GROUP BY t.person_gedcom_id
),

evidence AS (
  SELECT
    person_gedcom_id,
    total_facts,
    total_sources,
    avg_sources_per_fact,
    child_event_count,
    marriage_event_count,
    family_event_count,
    sourced_family_event_count
  FROM genealogy.gold_person_evidence_summary
),

proximity AS (
  SELECT
    p.person_id,
    MIN(ancestral_proximity)    AS proximity,
    MIN(d.generation_depth)     AS depth,
    d.person_id                 AS nearest_ancestor_id
  FROM genealogy.gold_ancestral_proximity p
  JOIN genealogy.gold_generation_depth d ON d.person_id = p.path_to_ancestor[0]
  GROUP BY p.person_id, d.person_id
),

census_coverage AS (
  SELECT
    t.person_gedcom_id,
    CASE WHEN t.birth_year IS NOT NULL AND t.birth_year <= 1840
          AND t.expected_end_year >= 1841 THEN 1 ELSE 0 END  AS expected_1841,
    CASE WHEN t.birth_year IS NOT NULL AND t.birth_year <= 1850
          AND t.expected_end_year >= 1851 THEN 1 ELSE 0 END  AS expected_1851,
    CASE WHEN t.birth_year IS NOT NULL AND t.birth_year <= 1860
          AND t.expected_end_year >= 1861 THEN 1 ELSE 0 END  AS expected_1861,
    CASE WHEN t.birth_year IS NOT NULL AND t.birth_year <= 1870
          AND t.expected_end_year >= 1871 THEN 1 ELSE 0 END  AS expected_1871,
    CASE WHEN t.birth_year IS NOT NULL AND t.birth_year <= 1880
          AND t.expected_end_year >= 1881 THEN 1 ELSE 0 END  AS expected_1881,
    CASE WHEN t.birth_year IS NOT NULL AND t.birth_year <= 1890
          AND t.expected_end_year >= 1891 THEN 1 ELSE 0 END  AS expected_1891,
    CASE WHEN t.birth_year IS NOT NULL AND t.birth_year <= 1900
          AND t.expected_end_year >= 1901 THEN 1 ELSE 0 END  AS expected_1901,
    CASE WHEN t.birth_year IS NOT NULL AND t.birth_year <= 1910
          AND t.expected_end_year >= 1911 THEN 1 ELSE 0 END  AS expected_1911,
    CASE WHEN t.birth_year IS NOT NULL AND t.birth_year <= 1920
          AND t.expected_end_year >= 1921 THEN 1 ELSE 0 END  AS expected_1921,
    CASE WHEN t.birth_year IS NOT NULL AND t.birth_year <= 1938
          AND t.expected_end_year >= 1939 THEN 1 ELSE 0 END  AS expected_1939
  FROM timeline t
),

uncovered_counts AS (
  SELECT person_gedcom_id, COUNT(*) AS uncovered_count
  FROM genealogy.gold_source_coverage
  WHERE coverage_status = 'UNCOVERED'
  GROUP BY person_gedcom_id
),

--TODO: Move burial count to gold_person_fact_summary
burial_events AS (
  SELECT DISTINCT person_gedcom_id
  FROM genealogy.gold_person_event_timeline
  WHERE event_type IN ('BURI', 'CREM')
),

ocr_signals AS (
  SELECT
    p.person_gedcom_id,
    CASE WHEN COUNT(sdp.file_id) = 0 THEN TRUE ELSE FALSE END
      AS no_documents,
    MAX(CASE WHEN sc.coverage_status = 'DOCUMENT_NO_TRANSCRIPT' THEN TRUE ELSE FALSE END)
      AS has_docs_not_transcribed,
    MAX(CASE WHEN fc.status = 'CONFLICT' THEN TRUE ELSE FALSE END)
      AS has_fact_conflict,
    MAX(CASE WHEN ot.doc_type_detected = 'MilitaryRecord' THEN TRUE ELSE FALSE END)
      AS has_military_record,
    MAX(CASE WHEN ot.doc_type_detected = 'NewspaperClipping' THEN TRUE ELSE FALSE END)
      AS has_newspaper,
    MAX(CASE WHEN ot.doc_type_detected IN ('Will', 'Probate') THEN TRUE ELSE FALSE END)
      AS has_will_probate,
    COUNT(DISTINCT CASE WHEN ot.doc_type_detected != 'BMDIndex'
                          AND ot.file_id IS NOT NULL THEN ot.file_id END)
      AS non_bmd_transcript_count
  FROM genealogy.gold_person_life p
  LEFT JOIN genealogy.silver_document_person sdp ON sdp.person_gedcom_id = p.person_gedcom_id
  LEFT JOIN genealogy.gold_source_coverage sc    ON sc.person_gedcom_id  = p.person_gedcom_id
  LEFT JOIN genealogy.gold_fact_comparison fc    ON fc.person_gedcom_id  = p.person_gedcom_id
  LEFT JOIN genealogy.ocr_transcriptions ot      ON ot.file_id = sdp.file_id
  GROUP BY p.person_gedcom_id
),

fact_comparison_signals AS (
  SELECT
    fc.person_gedcom_id,
    MAX(CASE WHEN fc.status = 'TRANSCRIPT_ONLY' THEN TRUE ELSE FALSE END)
      AS has_transcript_only_facts
  FROM genealogy.gold_fact_comparison fc
  GROUP BY fc.person_gedcom_id
),

-- Citation count for UNCOVERED_SOURCES applicability
citation_counts AS (
  SELECT person_gedcom_id, COUNT(*) AS citation_count
  FROM genealogy.gold_source_coverage
  GROUP BY person_gedcom_id
),

story_status AS (
  SELECT person_gedcom_id, story_written
  FROM genealogy.silver_person_story_status
)

SELECT
  p.person_gedcom_id,

  -- ── Context columns (NEW in v4 — needed by gold_person_signal_scores) ────
  p.birth_year,
  p.death_year,
  p.sex,
  p.expected_end_year,
  p.effective_span_years,
  p.event_count,
  fs.earliest_marriage_year,
  fs.num_child_births,
  fs.num_marriages,
  e.total_facts,
  -- Total expected census years (for MISSING_CENSUS_COVERAGE applicability)
  (
    cc.expected_1841 + cc.expected_1851 + cc.expected_1861 + cc.expected_1871 +
    cc.expected_1881 + cc.expected_1891 + cc.expected_1901 + cc.expected_1911 +
    cc.expected_1921 + cc.expected_1939
  )                                               AS total_expected_censuses,
  -- Transcript / citation flags (for evidence signal applicability)
  (COALESCE(o.non_bmd_transcript_count, 0) > 0)  AS has_any_transcript,
  (COALESCE(cit.citation_count, 0) > 0)          AS has_citations,

  -- ── Structural / lineage context ─────────────────────────────────────────
  CASE WHEN p.person_gedcom_id = pr.nearest_ancestor_id THEN TRUE ELSE FALSE END
    AS is_direct_ancestor,
  pr.depth,
  pr.proximity,

  CASE WHEN pr.proximity = 0 THEN TRUE ELSE FALSE END   AS SIGNAL_DIRECT_ANCESTOR,
  CASE WHEN pr.proximity = 1 THEN TRUE ELSE FALSE END   AS SIGNAL_CLOSE_COLLATERAL,

  -- ── INTEGRITY — Completeness ──────────────────────────────────────────────

  CASE WHEN p.birth_year IS NULL
    THEN TRUE ELSE FALSE END AS SIGNAL_NO_BIRTH_RECORDED,

  CASE WHEN p.death_year IS NULL
    AND p.expected_end_year < year(current_date)
    AND (p.birth_year IS NULL OR p.birth_year <= 1930)
    THEN TRUE ELSE FALSE END AS SIGNAL_NO_DEATH_RECORDED,

  CASE WHEN fs.num_marriages = 0
    AND (p.death_year IS NULL
      OR (p.death_year - COALESCE(p.birth_year, p.death_year - 40)) >= 16)
    THEN TRUE ELSE FALSE END AS SIGNAL_NO_MARRIAGES,

  CASE WHEN fs.num_child_births = 0
    AND COALESCE(pr.proximity, 99) <= 1
    AND NOT (p.death_year IS NOT NULL AND p.effective_span_years BETWEEN 16 AND 40)
    THEN TRUE ELSE FALSE END AS SIGNAL_NO_CHILDREN,

  CASE WHEN fs.num_parents < 2
    THEN TRUE ELSE FALSE END AS SIGNAL_MISSING_PARENT,

  CASE WHEN (
      (cc.expected_1841 > 0 AND fs.has_1841_census = 0) OR
      (cc.expected_1851 > 0 AND fs.has_1851_census = 0) OR
      (cc.expected_1861 > 0 AND fs.has_1861_census = 0) OR
      (cc.expected_1871 > 0 AND fs.has_1871_census = 0) OR
      (cc.expected_1881 > 0 AND fs.has_1881_census = 0) OR
      (cc.expected_1891 > 0 AND fs.has_1891_census = 0) OR
      (cc.expected_1901 > 0 AND fs.has_1901_census = 0) OR
      (cc.expected_1911 > 0 AND fs.has_1911_census = 0) OR
      (cc.expected_1921 > 0 AND fs.has_1921_census = 0) OR
      (cc.expected_1939 > 0 AND fs.has_1939_register = 0)
    ) THEN TRUE ELSE FALSE END AS SIGNAL_MISSING_CENSUS_COVERAGE,

  CASE WHEN COALESCE(uc.uncovered_count, 0) > 1
    THEN TRUE ELSE FALSE END AS SIGNAL_UNCOVERED_SOURCES,

  COALESCE(o.has_docs_not_transcribed, FALSE) AS SIGNAL_DOCS_NOT_TRANSCRIBED,

  CASE WHEN p.death_year IS NULL
    AND p.last_event_year < p.birth_year + 40
    THEN TRUE ELSE FALSE END AS SIGNAL_LATE_LIFE_GAP,

  CASE WHEN p.max_event_age <= 25 AND e.family_event_count > 0
    THEN TRUE ELSE FALSE END AS SIGNAL_EARLY_LIFE_ONLY,

  CASE WHEN fs.max_days_between_child_births > 730
    OR (fs.num_marriages > 0 AND fs.num_child_births = 0)
    THEN TRUE ELSE FALSE END AS SIGNAL_CHILD_GAPS,

  CASE WHEN p.birth_year BETWEEN 1880 AND 1928
    AND fs.num_military = 0
    AND NOT COALESCE(o.has_military_record, FALSE)
    THEN TRUE ELSE FALSE END AS SIGNAL_UNCONFIRMED_MILITARY,

  CASE WHEN p.sex = 'M'
    AND fs.num_occupations = 0
    AND p.birth_year IS NOT NULL
    AND p.expected_end_year >= p.birth_year + 18
    THEN TRUE ELSE FALSE END AS SIGNAL_MISSING_OCCUPATION,

  CASE WHEN p.death_year IS NOT NULL
    AND p.death_year >= 1837
    AND be.person_gedcom_id IS NULL
    THEN TRUE ELSE FALSE END AS SIGNAL_MISSING_BURIAL,

  CASE WHEN p.effective_span_years >= 20
    AND fs.num_residences = 0
    THEN TRUE ELSE FALSE END AS SIGNAL_NO_RESIDENCE,

  -- ── INTEGRITY — Evidence ──────────────────────────────────────────────────

  COALESCE(o.no_documents, FALSE)                    AS SIGNAL_NO_DOCUMENTS_AT_ALL,
  COALESCE(fcs.has_transcript_only_facts, FALSE)     AS SIGNAL_TRANSCRIPT_ONLY_FACTS,
  COALESCE(o.has_fact_conflict, FALSE)               AS SIGNAL_FACT_CONFLICT,

  CASE WHEN e.avg_sources_per_fact IS NULL OR e.avg_sources_per_fact < 0.3
    THEN TRUE ELSE FALSE END AS SIGNAL_VERY_LOW_EVIDENCE_DENSITY,

  CASE WHEN COALESCE(e.avg_sources_per_fact, 0) < 1.0
    THEN TRUE ELSE FALSE END AS SIGNAL_LOW_EVIDENCE_DENSITY,

  CASE WHEN e.total_sources = 1 AND e.total_facts >= 3
    THEN TRUE ELSE FALSE END AS SIGNAL_SINGLE_SOURCE_DEPENDENCE,

  CASE WHEN e.family_event_count > e.sourced_family_event_count
    THEN TRUE ELSE FALSE END AS SIGNAL_UNSOURCED_FAMILY_EVENTS,

  CASE WHEN fs.has_given_name = 0 OR fs.has_surname = 0
    THEN TRUE ELSE FALSE END AS SIGNAL_INCOMPLETE_NAME,

  CASE WHEN (
      (p.birth_year  >= 1837 AND fs.birth_date_precision  <> 'DAY')
      OR (p.death_year  >= 1837 AND fs.death_date_precision  <> 'DAY')
      OR (fs.marriage_date_precision IS NOT NULL
          AND fs.earliest_marriage_year >= 1837
          AND fs.marriage_date_precision <> 'DAY')
    ) THEN TRUE ELSE FALSE END AS SIGNAL_IMPRECISE_DATES,

  CASE WHEN p.birth_year IS NOT NULL AND p.birth_year >= 1600
    AND (
      (p.birth_year IS NOT NULL AND (
        fs.birth_place IS NULL
        OR LENGTH(fs.birth_place) - LENGTH(REPLACE(fs.birth_place, ',', '')) < 1))
      OR (p.death_year IS NOT NULL AND (
        fs.death_place IS NULL
        OR LENGTH(fs.death_place) - LENGTH(REPLACE(fs.death_place, ',', '')) < 1))
    ) THEN TRUE ELSE FALSE END AS SIGNAL_IMPRECISE_PLACES,

  -- ── NARRATIVE — Texture ───────────────────────────────────────────────────

  CASE WHEN fs.num_family_units > 1
    THEN TRUE ELSE FALSE END AS SIGNAL_MULTIPLE_SPOUSES,

  CASE WHEN p.death_year IS NOT NULL
    AND p.effective_span_years BETWEEN 16 AND 40
    THEN TRUE ELSE FALSE END AS SIGNAL_YOUNG_DEATH,

  CASE WHEN fs.num_migration > 0 OR fs.num_countries > 1
    THEN TRUE ELSE FALSE END AS SIGNAL_MIGRANT,

  CASE WHEN fs.num_military > 0 OR COALESCE(o.has_military_record, FALSE)
    THEN TRUE ELSE FALSE END AS SIGNAL_CONFIRMED_MILITARY,

  CASE WHEN fs.num_child_births >= 6
    THEN TRUE ELSE FALSE END AS SIGNAL_LARGE_FAMILY,

  COALESCE(o.has_newspaper, FALSE)   AS SIGNAL_NEWSPAPER_MENTION,
  COALESCE(o.has_will_probate, FALSE) AS SIGNAL_WILL_OR_PROBATE,
  COALESCE(ss.story_written, FALSE)  AS SIGNAL_STORY_WRITTEN,

  -- ── NARRATIVE — Context ───────────────────────────────────────────────────

  CASE WHEN COALESCE(o.non_bmd_transcript_count, 0) >= 2
    THEN TRUE ELSE FALSE END AS SIGNAL_TRANSCRIPT_RICH,

  CASE WHEN fs.num_occupations > 1
    THEN TRUE ELSE FALSE END AS SIGNAL_VARIED_OCCUPATIONS,

  CASE WHEN (fs.num_marriages > 0 OR fs.num_child_births > 0 OR
    fs.has_1841_census = 0 OR
    fs.has_1851_census = 0 OR
    fs.has_1861_census = 0 OR
    fs.has_1871_census = 0 OR
    fs.has_1881_census = 0 OR
    fs.has_1891_census = 0 OR
    fs.has_1901_census = 0 OR
    fs.has_1911_census = 0 OR
    fs.has_1921_census = 0 OR
    fs.has_1939_register = 0)
    AND fs.num_residences = 0
    THEN TRUE ELSE FALSE END AS SIGNAL_POSSIBLE_RESIDENCE,

  CASE WHEN p.sex = 'F' AND fs.has_1939_register = 1 AND fs.num_marriages = 0
    THEN TRUE ELSE FALSE END AS SIGNAL_POSSIBLE_MARRIAGE,

  CASE WHEN p.sex = 'F' AND fs.has_1911_census = 1 AND p.birth_year <= 1895
    THEN TRUE ELSE FALSE END AS SIGNAL_POSSIBLE_CHILDREN

FROM timeline p
JOIN  evidence e                        ON p.person_gedcom_id = e.person_gedcom_id
JOIN  genealogy.gold_person_fact_summary fs ON fs.person_gedcom_id = p.person_gedcom_id
LEFT JOIN proximity pr                  ON p.person_gedcom_id = pr.person_id
LEFT JOIN census_coverage cc            ON p.person_gedcom_id = cc.person_gedcom_id
LEFT JOIN uncovered_counts uc           ON p.person_gedcom_id = uc.person_gedcom_id
LEFT JOIN burial_events be              ON p.person_gedcom_id = be.person_gedcom_id
LEFT JOIN ocr_signals o                 ON p.person_gedcom_id = o.person_gedcom_id
LEFT JOIN fact_comparison_signals fcs   ON p.person_gedcom_id = fcs.person_gedcom_id
LEFT JOIN citation_counts cit           ON p.person_gedcom_id = cit.person_gedcom_id
LEFT JOIN story_status ss               ON p.person_gedcom_id = ss.person_gedcom_id;


In [0]:

%sql
-- ============================================================
-- CELL 2: Signal fire rates
-- Quick sanity check — see notebook_calibration for guidance ranges.
-- ============================================================

SELECT
  signal_code,
  fire_count,
  ROUND(fire_count * 100.0 / total_people, 1) AS fire_pct
FROM (
  SELECT
    counts.signal_code,
    counts.fire_count,
    totals.total_people
  FROM (
    SELECT 'SIGNAL_NO_BIRTH_RECORDED'       AS signal_code, SUM(CASE WHEN SIGNAL_NO_BIRTH_RECORDED       THEN 1 ELSE 0 END) AS fire_count FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_NO_DEATH_RECORDED',       SUM(CASE WHEN SIGNAL_NO_DEATH_RECORDED       THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_NO_MARRIAGES',            SUM(CASE WHEN SIGNAL_NO_MARRIAGES            THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_NO_CHILDREN',             SUM(CASE WHEN SIGNAL_NO_CHILDREN             THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_MISSING_PARENT',          SUM(CASE WHEN SIGNAL_MISSING_PARENT          THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_MISSING_CENSUS_COVERAGE', SUM(CASE WHEN SIGNAL_MISSING_CENSUS_COVERAGE THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_UNCOVERED_SOURCES',       SUM(CASE WHEN SIGNAL_UNCOVERED_SOURCES       THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_DOCS_NOT_TRANSCRIBED',    SUM(CASE WHEN SIGNAL_DOCS_NOT_TRANSCRIBED    THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_LATE_LIFE_GAP',           SUM(CASE WHEN SIGNAL_LATE_LIFE_GAP           THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_EARLY_LIFE_ONLY',         SUM(CASE WHEN SIGNAL_EARLY_LIFE_ONLY         THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_CHILD_GAPS',              SUM(CASE WHEN SIGNAL_CHILD_GAPS              THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_UNCONFIRMED_MILITARY',    SUM(CASE WHEN SIGNAL_UNCONFIRMED_MILITARY    THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_MISSING_OCCUPATION',      SUM(CASE WHEN SIGNAL_MISSING_OCCUPATION      THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_MISSING_BURIAL',          SUM(CASE WHEN SIGNAL_MISSING_BURIAL          THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_NO_RESIDENCE',            SUM(CASE WHEN SIGNAL_NO_RESIDENCE            THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_NO_DOCUMENTS_AT_ALL',     SUM(CASE WHEN SIGNAL_NO_DOCUMENTS_AT_ALL     THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_TRANSCRIPT_ONLY_FACTS',   SUM(CASE WHEN SIGNAL_TRANSCRIPT_ONLY_FACTS   THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_FACT_CONFLICT',           SUM(CASE WHEN SIGNAL_FACT_CONFLICT           THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_VERY_LOW_EVIDENCE_DENSITY',SUM(CASE WHEN SIGNAL_VERY_LOW_EVIDENCE_DENSITY THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_LOW_EVIDENCE_DENSITY',    SUM(CASE WHEN SIGNAL_LOW_EVIDENCE_DENSITY    THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_SINGLE_SOURCE_DEPENDENCE',SUM(CASE WHEN SIGNAL_SINGLE_SOURCE_DEPENDENCE THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_UNSOURCED_FAMILY_EVENTS', SUM(CASE WHEN SIGNAL_UNSOURCED_FAMILY_EVENTS THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_INCOMPLETE_NAME',         SUM(CASE WHEN SIGNAL_INCOMPLETE_NAME         THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_IMPRECISE_DATES',         SUM(CASE WHEN SIGNAL_IMPRECISE_DATES         THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_IMPRECISE_PLACES',        SUM(CASE WHEN SIGNAL_IMPRECISE_PLACES        THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_MIGRANT',                 SUM(CASE WHEN SIGNAL_MIGRANT                 THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_CONFIRMED_MILITARY',      SUM(CASE WHEN SIGNAL_CONFIRMED_MILITARY      THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_MULTIPLE_SPOUSES',        SUM(CASE WHEN SIGNAL_MULTIPLE_SPOUSES        THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_YOUNG_DEATH',             SUM(CASE WHEN SIGNAL_YOUNG_DEATH             THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_LARGE_FAMILY',            SUM(CASE WHEN SIGNAL_LARGE_FAMILY            THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_NEWSPAPER_MENTION',       SUM(CASE WHEN SIGNAL_NEWSPAPER_MENTION       THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_WILL_OR_PROBATE',         SUM(CASE WHEN SIGNAL_WILL_OR_PROBATE         THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_STORY_WRITTEN',           SUM(CASE WHEN SIGNAL_STORY_WRITTEN           THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_TRANSCRIPT_RICH',         SUM(CASE WHEN SIGNAL_TRANSCRIPT_RICH         THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_VARIED_OCCUPATIONS',      SUM(CASE WHEN SIGNAL_VARIED_OCCUPATIONS      THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_POSSIBLE_RESIDENCE',      SUM(CASE WHEN SIGNAL_POSSIBLE_RESIDENCE      THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_POSSIBLE_MARRIAGE',       SUM(CASE WHEN SIGNAL_POSSIBLE_MARRIAGE       THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals UNION ALL
    SELECT 'SIGNAL_POSSIBLE_CHILDREN',       SUM(CASE WHEN SIGNAL_POSSIBLE_CHILDREN       THEN 1 ELSE 0 END) FROM genealogy.gold_research_person_signals
  ) counts
  CROSS JOIN (SELECT COUNT(DISTINCT person_gedcom_id) AS total_people FROM genealogy.gold_research_person_signals) totals
)
ORDER BY fire_pct DESC;


In [0]:

%sql
-- UNCONFIRMED and CONFIRMED military must be mutually exclusive
SELECT COUNT(*) AS violations,
       'UNCONFIRMED and CONFIRMED both TRUE' AS check_name
FROM genealogy.gold_research_person_signals
WHERE SIGNAL_UNCONFIRMED_MILITARY = TRUE AND SIGNAL_CONFIRMED_MILITARY = TRUE

UNION ALL

-- Confirm new context columns are populated
SELECT COUNT(*) AS non_null_birth_year, 'birth_year populated' AS check_name
FROM genealogy.gold_research_person_signals
WHERE birth_year IS NOT NULL;
